# Start Trek — PPO sur LunarLander-v3

**Objectif :** entraîner un deuxième agent avec **PPO (Proximal Policy Optimization)**, puis le comparer à l'agent DQN.
L'environnement est résolu avec un **score moyen ≥ 200 sur 100 épisodes**.

**DQN vs PPO :** le DQN apprend une **note** (Q-value) pour chaque action et joue la meilleure.
Le PPO apprend directement une **politique** : une **probabilité** pour chaque action, et il tire son action au sort.
L'exploration est donc intégrée : pas besoin d'epsilon-greedy.

**Plan**
1. L'environnement
2. Acteur et critique
3. Rollout storage
4. Avantages (GAE)
5. Collecte d'un rollout

*À venir : loss PPO (3 termes), hyperparamètres, boucle d'entraînement, courbes, évaluation sur 100 épisodes, comparaison avec le DQN.*

In [1]:
import random
from collections import Counter

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

## 1. L'environnement

Comme dans le notebook DQN : pas d'affichage pendant l'entraînement, et la taille de l'état et le nombre d'actions
sont lus depuis l'environnement.

In [2]:
ENV_ID = "LunarLander-v3"


def make_env(render_mode=None):
    return gym.make(ENV_ID, render_mode=render_mode)


env = make_env()
STATE_DIM = env.observation_space.shape[0]
N_ACTIONS = env.action_space.n
env.close()

print("Taille de l'état :", STATE_DIM)
print("Nombre d'actions :", N_ACTIONS)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

Taille de l'état : 8
Nombre d'actions : 4


### Pourquoi un épisode se termine

Mêmes causes que pour le DQN : `crash`, `out` (sortie de l'écran), `rest` (posé) → **terminated** ;
limite de 1000 steps → **truncated**.

In [3]:
END_REASONS = ["crash", "out", "rest", "truncated"]


def end_reason(env, state, terminated, truncated):
    if terminated:
        if env.unwrapped.game_over:
            return "crash"
        if abs(state[0]) >= 1.0:
            return "out"
        return "rest"
    if truncated:
        return "truncated"
    return None

## 2. Acteur et critique

| Élément | Rôle |
|---|---|
| **Acteur** | Donne la **politique** : une probabilité pour chaque action dans un état donné. |
| **Critique** | Estime la **valeur** $V(s)$ d'un état : le total de points que l'agent peut espérer à partir de cet état, en suivant sa politique actuelle. |

**Notre choix : deux réseaux séparés**, `8 → 64 → 64 → 4` pour l'acteur et `8 → 64 → 64 → 1` pour le critique, avec Tanh.
- C'est le plus simple à comprendre et à régler : l'erreur du critique ne vient pas perturber l'acteur.
- Un réseau partagé avec deux sorties (« têtes ») économiserait des paramètres, mais il faudrait bien équilibrer
  les deux losses. Sur un état de 8 valeurs, deux petits réseaux ne coûtent presque rien.

L'acteur sort des **logits** (des scores), transformés en probabilités par une distribution `Categorical`.
Pour jouer, on **tire une action au sort** selon ces probabilités : c'est ce qui fait explorer l'agent.

In [4]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, n_actions),
        )
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def get_value(self, states):
        return self.critic(states).squeeze(-1)

    def get_action(self, state):
        # Tire une action au sort selon la politique, pendant la collecte
        state = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            dist = Categorical(logits=self.actor(state))
            action = dist.sample()
            value = self.get_value(state)
        return int(action), float(dist.log_prob(action)), float(value)

    def evaluate(self, states, actions):
        # Pendant l'apprentissage : probabilité des actions jouées, entropie et valeurs
        dist = Categorical(logits=self.actor(states))
        return dist.log_prob(actions), dist.entropy(), self.get_value(states)

**Test :** avec un réseau pas encore entraîné, les 4 probabilités doivent être à peu près égales (≈ 25 %)
et leur somme doit faire 1.

In [5]:
set_seed(0)
model = ActorCritic(STATE_DIM, N_ACTIONS)
print(model)

env = make_env()
state, _ = env.reset(seed=0)
env.close()

with torch.no_grad():
    probs = torch.softmax(model.actor(torch.tensor(state)), dim=-1)
    value = model.get_value(torch.tensor(state))
print("Probabilités des 4 actions :", probs.numpy().round(3), "| somme :", round(float(probs.sum()), 3))
print("Valeur estimée de l'état :", round(float(value), 3))
print("10 actions tirées au sort :", [model.get_action(state)[0] for _ in range(10)])

ActorCritic(
  (actor): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=4, bias=True)
  )
  (critic): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)
Probabilités des 4 actions : [0.291 0.229 0.232 0.248] | somme : 1.0
Valeur estimée de l'état : -0.144
10 actions tirées au sort : [1, 1, 1, 0, 2, 3, 2, 1, 0, 0]


## 3. Rollout storage

Le PPO **n'utilise pas de replay buffer** : il ne peut apprendre que sur des expériences jouées par **sa politique actuelle**.
Il joue donc un **rollout** (un paquet de steps), apprend dessus quelques fois, puis **le jette** et recommence.

Par rapport au replay buffer du DQN, on stocke en plus, pour chaque step :
- `log_prob` : le log de la probabilité de l'action jouée (pour calculer le **ratio** entre nouvelle et ancienne politique) ;
- `value` : la valeur $V(s)$ donnée par le critique (pour calculer les **avantages**).

`get()` renvoie les éléments **groupés en tenseurs**, et `clear()` vide le stockage après chaque mise à jour.

In [6]:
class RolloutStorage:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states, self.actions, self.log_probs = [], [], []
        self.values, self.rewards, self.dones = [], [], []

    def add(self, state, action, log_prob, value, reward, done):
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.rewards.append(reward)
        self.dones.append(done)

    def get(self):
        return (torch.tensor(np.array(self.states), dtype=torch.float32),
                torch.tensor(self.actions, dtype=torch.int64),
                torch.tensor(self.log_probs, dtype=torch.float32),
                torch.tensor(self.values, dtype=torch.float32),
                torch.tensor(self.rewards, dtype=torch.float32),
                torch.tensor(self.dones, dtype=torch.float32))

    def __len__(self):
        return len(self.states)

### Terminated, truncated et `done`

Dans le rollout, `done = 1` veut dire « l'épisode s'arrête ici » : il ne faut pas mélanger cet épisode avec le suivant
quand on calcule les avantages. Mais les deux cas sont traités **différemment** :
- **terminated** (crash, sortie, posé) : il n'y a vraiment plus rien après, la valeur future est 0 ;
- **truncated** (coupé à 1000 steps) : le module aurait pu continuer. On ajoute donc à la dernière récompense
  la valeur estimée de la suite : $r \leftarrow r + \gamma \, V(s_{\text{final}})$.

## 4. Avantages (GAE)

L'**avantage** $A_t$ dit si une action a été **meilleure ou moins bonne que prévu** par le critique.
Positif : l'action doit devenir plus probable. Négatif : moins probable.

On utilise la **Generalized Advantage Estimation (GAE)**, en partant de la fin du rollout :

$$\delta_t = r_t + \gamma \, (1 - done_t) \, V(s_{t+1}) - V(s_t)$$

$$A_t = \delta_t + \gamma \lambda \, (1 - done_t) \, A_{t+1}$$

- $\delta_t$ : l'erreur du critique sur un seul step (« ce qui s'est passé » moins « ce qui était prévu ») ;
- $\lambda$ règle le compromis : $\lambda = 0$ ne regarde qu'un step (peu de bruit mais biaisé),
  $\lambda = 1$ regarde jusqu'à la fin de l'épisode (pas biaisé mais bruité). On prendra $\lambda = 0.95$.

Les **returns** (cibles du critique) sont $R_t = A_t + V(s_t)$.

In [7]:
def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    advantages = torch.zeros(len(rewards))
    next_advantage = 0.0
    next_value = last_value
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * (1 - dones[t]) * next_value - values[t]
        next_advantage = delta + gamma * lam * (1 - dones[t]) * next_advantage
        advantages[t] = next_advantage
        next_value = values[t]
    returns = advantages + values
    return advantages, returns

**Test à la main :** 3 steps de récompense 1, un critique qui prédit toujours 0, épisode fini au 3e step,
γ = 0.99 et λ = 1. Les returns doivent être la somme des récompenses futures :
1 + 0.99 + 0.99² = 2.9701, puis 1 + 0.99 = 1.99, puis 1.

In [8]:
rewards = torch.tensor([1.0, 1.0, 1.0])
values = torch.tensor([0.0, 0.0, 0.0])
dones = torch.tensor([0.0, 0.0, 1.0])
advantages, returns = compute_gae(rewards, values, dones, last_value=0.0, gamma=0.99, lam=1.0)
print("Returns :", returns.numpy().round(4))

Returns : [2.9701 1.99   1.    ]


## 5. Collecte d'un rollout

L'agent joue `n_steps` steps avec sa politique actuelle et remplit le rollout storage.
Si un épisode se termine au milieu, on recommence un nouvel épisode et on continue.
À la fin, on garde l'état où on s'est arrêté, pour reprendre au même endroit au rollout suivant.

In [9]:
def collect_rollout(env, model, storage, state, n_steps, gamma=0.99, ongoing=None):
    # ongoing = [score, longueur] de l'épisode en cours : il a pu commencer pendant le rollout précédent
    if ongoing is None:
        ongoing = [0.0, 0]
    finished = []  # un élément par épisode terminé : (score, longueur, cause de fin, step dans le rollout)

    for step in range(1, n_steps + 1):
        action, log_prob, value = model.get_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        ongoing[0] += reward
        ongoing[1] += 1

        if truncated and not terminated:
            # Épisode coupé : on ajoute la valeur estimée de la suite
            with torch.no_grad():
                reward += gamma * float(model.get_value(torch.tensor(next_state, dtype=torch.float32)))

        storage.add(state, action, log_prob, value, reward, float(terminated or truncated))

        if terminated or truncated:
            finished.append((ongoing[0], ongoing[1], end_reason(env, next_state, terminated, truncated), step))
            ongoing[0], ongoing[1] = 0.0, 0
            state, _ = env.reset()
        else:
            state = next_state

    with torch.no_grad():
        last_value = float(model.get_value(torch.tensor(state, dtype=torch.float32)))
    return state, last_value, finished

**Test :** on collecte un rollout de 2048 steps avec l'agent non entraîné, puis on calcule les avantages.

In [10]:
set_seed(0)
model = ActorCritic(STATE_DIM, N_ACTIONS)
storage = RolloutStorage()

env = make_env()
state, _ = env.reset(seed=0)
state, last_value, episodes = collect_rollout(env, model, storage, state, n_steps=2048)
env.close()

states, actions, log_probs, values, rewards, dones = storage.get()
advantages, returns = compute_gae(rewards, values, dones, last_value)

print("Steps dans le rollout :", len(storage))
print("Forme des états :", tuple(states.shape))
print("Épisodes terminés pendant le rollout :", len(episodes),
      "| score moyen :", round(np.mean([e[0] for e in episodes]), 1))
print("Causes de fin :", Counter(e[2] for e in episodes))
print("Avantages : moyenne", round(float(advantages.mean()), 2), "| écart-type", round(float(advantages.std()), 2))

storage.clear()
print("Après clear() :", len(storage), "step")

Steps dans le rollout : 2048
Forme des états : (2048, 8)
Épisodes terminés pendant le rollout : 23 | score moyen : -188.2
Causes de fin : Counter({'crash': 20, 'out': 3})
Avantages : moyenne -33.22 | écart-type 29.61
Après clear() : 0 step


## Prochaine étape

Les briques du PPO sont en place : acteur, critique, rollout storage, avantages (GAE) et collecte d'un rollout.
La suite :
- la **loss PPO** et ses trois termes : la loss de politique **clippée**, la loss de valeur (MSE) et le **bonus d'entropie** ;
- la **boucle d'entraînement** : collecter un rollout, apprendre dessus pendant quelques epochs en mini-batchs, le jeter, recommencer ;
- les **courbes**, l'**évaluation sur 100 épisodes**, plusieurs seeds, une expérience, et la **comparaison avec le DQN**.